# 🤖 Notebook 4: Model Building

Train 4 classifiers: Logistic Regression, Decision Tree, Random Forest, Gradient Boosting.

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, roc_auc_score
import joblib, warnings, os
warnings.filterwarnings('ignore')

df = pd.read_csv('../data/processed/cleaned_churn_data.csv')
print(f"Dataset: {df.shape}")

## 4.1 Prepare Feature Matrix

In [ ]:
# Binary encode
for col in ['Partner','Dependents','PhoneService','PaperlessBilling']:
    df[col] = df[col].map({'Yes':1,'No':0})
df['gender'] = df['gender'].map({'Male':1,'Female':0})

# One-hot encode categoricals
cats = ['InternetService','Contract','PaymentMethod','MultipleLines',
        'OnlineSecurity','OnlineBackup','DeviceProtection','TechSupport',
        'StreamingTV','StreamingMovies']
df_enc = pd.get_dummies(df, columns=cats).fillna(0)

X = df_enc.drop(columns=['Churn','customerID','TenureCohort'], errors='ignore')
y = df_enc['Churn']

# Scale
scaler = StandardScaler()
X_scaled = X.copy()
for c in ['tenure','MonthlyCharges','TotalCharges']:
    X_scaled[c] = scaler.fit_transform(X[[c]])

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train: {X_train.shape[0]:,} | Test: {X_test.shape[0]:,} | Features: {X_train.shape[1]}")

## 4.2 Logistic Regression

In [ ]:
lr = LogisticRegression(solver='lbfgs', max_iter=1000, random_state=42)
lr.fit(X_train, y_train)
lr_acc = accuracy_score(y_test, lr.predict(X_test))
lr_auc = roc_auc_score(y_test, lr.predict_proba(X_test)[:,1])
cv_lr  = cross_val_score(lr, X_scaled, y, cv=5, scoring='roc_auc').mean()
print(f"Logistic Regression  —  Acc: {lr_acc:.4f}  AUC: {lr_auc:.4f}  CV-AUC: {cv_lr:.4f}")

## 4.3 Decision Tree

In [ ]:
dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train, y_train)
dt_acc = accuracy_score(y_test, dt.predict(X_test))
dt_auc = roc_auc_score(y_test, dt.predict_proba(X_test)[:,1])
print(f"Decision Tree        —  Acc: {dt_acc:.4f}  AUC: {dt_auc:.4f}")

## 4.4 Random Forest

In [ ]:
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
rf_acc = accuracy_score(y_test, rf.predict(X_test))
rf_auc = roc_auc_score(y_test, rf.predict_proba(X_test)[:,1])
print(f"Random Forest        —  Acc: {rf_acc:.4f}  AUC: {rf_auc:.4f}")

## 4.5 Gradient Boosting

In [ ]:
gb = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42)
gb.fit(X_train, y_train)
gb_acc = accuracy_score(y_test, gb.predict(X_test))
gb_auc = roc_auc_score(y_test, gb.predict_proba(X_test)[:,1])
print(f"Gradient Boosting    —  Acc: {gb_acc:.4f}  AUC: {gb_auc:.4f}")

## 4.6 Save Models

In [ ]:
os.makedirs('../models/trained_models', exist_ok=True)
for name, model in [('logistic_regression',lr),('decision_tree',dt),
                    ('random_forest',rf),('gradient_boosting',gb)]:
    joblib.dump(model, f'../models/trained_models/{name}.pkl')
    print(f"Saved: {name}.pkl")